# Week 1 — Digital Rock 데이터 탐색 (2조 Intro)

## 이번 주 학습 목표
1. **3D voxel 데이터**가 무엇인지 이해하고, binary 파일에서 직접 불러올 수 있다.
2. 부피 데이터를 **세 방향(z, y, x) 단면(slice)** 으로 잘라 시각화할 수 있다.
3. **공극률(porosity)** 의 정의를 알고 직접 계산할 수 있다.
4. **Sparse imaging** 시나리오를 시뮬레이션하고, 본 연구가 풀려는 문제를 자신의 말로 정의할 수 있다.
5. **단순 평균 보간(linear interpolation)** 을 시각적으로 이해한다.

## 학습 방식
**본 코스는 "코드를 직접 짜는 것" 보다 "배포된 코드의 인자(parameter)를 바꿔보며 결과 변화를 관찰하는 것" 을 중심으로 합니다.**

- 각 섹션마다 **[Try-it!]** 박스가 있습니다. 변수 값을 다양하게 바꿔보세요.
- **[보조 설명]** 박스는 모르는 용어/개념을 풀어 설명합니다.
- **[해석 질문]** 박스는 결과를 자신의 말로 답하는 연습입니다.
- 마지막 **탐구 과제** 는 노트북 안에서 코드를 수정하면서 답해주세요.

## 0. 환경 준비

**시작 전 확인:**
- 가상환경 `rock` 이 활성화되어 있는지 (`conda activate rock`)
- `COMMON/environment_setup_guide.md` 설치 완료
- 이 notebook은 `group2_intro/week1/notebooks/` 에 있다는 가정으로 경로가 설정됨

> **[보조 설명]** `import` 는 "이 라이브러리/모듈을 사용하겠다" 는 선언입니다.
> `as np` 는 별명입니다 — `numpy.array(...)` 대신 `np.array(...)` 라고 짧게 쓸 수 있게 해줍니다.

In [ ]:
import sys
from pathlib import Path

# helpers 모듈 import 경로 추가
sys.path.insert(0, str(Path('..').resolve() / 'helpers'))

import numpy as np
import matplotlib.pyplot as plt

from dr_utils import (
    load_volume, porosity, porosity_profile,
    show_slice, show_three_axis,
    make_sparse, time_saving_ratio,
    linear_interpolate_slice,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)

setup_plot_style()
print('환경 준비 완료')

## 1. Digital Rock 데이터란?

**Micro-CT(마이크로 단층촬영)** 는 작은 암석 시료를 여러 각도에서 X-ray로 촬영해 **3D 부피 영상**을 만드는 기술입니다.
- 2D 사진 = pixel 격자 → shape: (높이, 너비)
- 3D 부피 = **voxel 격자** → shape: (깊이, 높이, 너비) = (Z, Y, X)

이번 주 다룰 데이터:
| 도메인 | shape | voxel 크기 | 값 |
|---|---|---|---|
| BB 사암 | 256×256×256 | 2.25 μm | 0=solid, 1=pore |
| CastleGate 사암 | 256×256×256 | 2.25 μm | 0=solid, 1=pore |

> **[보조 설명]** "binary segmentation 완료" 라 함은 원래의 grayscale CT 영상을 "공극 vs 암석" 두 가지로 분류 완료한 상태라는 뜻입니다. 그래서 값이 0과 1 뿐입니다.

In [ ]:
DATA_DIR = Path('..') / 'data'

bb = load_volume(DATA_DIR / 'BB_256.bin')
cg = load_volume(DATA_DIR / 'CastleGate_256.bin')

print(f'BB        : shape={bb.shape}, dtype={bb.dtype}, 고유값={np.unique(bb).tolist()}')
print(f'CastleGate: shape={cg.shape}, dtype={cg.dtype}, 고유값={np.unique(cg).tolist()}')
print(f'총 voxel 수: {bb.size:,}')

> **[자기 점검]** 출력에서 `고유값=[0, 1]` 이 나왔나요?
> 그렇다면 이 데이터는 "이진(binary) 부피" — 0은 암석, 1은 공극입니다.
>
> 만약 grayscale CT 였다면 어떤 값들이 나왔을까요? (힌트: uint8은 0~255)

## 2. 슬라이스 시각화 — 한 장씩 자세히

3D 부피는 한 번에 보기 어렵습니다. **한 단면(slice)** 만 잘라 봅시다.

- z축 slice: `volume[z, :, :]` → XY 평면
- y축 slice: `volume[:, y, :]` → ZX 평면
- x축 slice: `volume[:, :, x]` → ZY 평면

In [ ]:
# 가장 단순한 시각화 — z=128 슬라이스, 회색 cmap
show_slice(bb, axis=0, idx=128, cmap='gray', title='BB z=128')
plt.show()

> **[Try-it! ①]** 위 셀의 함수 인자들을 다음과 같이 바꿔보고 결과를 관찰하세요:
> - `axis=0, 1, 2` → 세 방향 단면이 어떻게 다른가?
> - `idx=0, 64, 128, 192, 255` → 가장자리와 중앙은 어떻게 다른가?
> - `cmap='gray', 'hot', 'viridis', 'plasma'` → 어느 cmap이 pore를 가장 잘 보여주는가?
>
> 아래 빈 셀에 위 변형을 자유롭게 시도해보세요.

In [ ]:
# [Try-it! ①] 직접 변형해보세요
show_slice(bb, axis=0, idx=128, cmap='gray')
plt.show()

# 예시: axis=1, idx=0 (가장자리)
show_slice(bb, axis=1, idx=0, cmap='gray')
plt.show()

## 3. 세 축 동시 비교

이제 세 축을 한 번에 봅시다. 본 연구의 핵심 contribution 중 하나가 "**세 방향 보간 결과 통합 (tri-axis aggregation)**" 입니다.
왜 세 방향을 모두 보는 게 좋을지 직접 확인해봅시다.

> **[보조 설명: tri-axis aggregation]** 영어로 tri = 셋, axis = 축, aggregation = 통합/합산.
> 즉 "세 축에서 각각 한 결과를 만들고 그것을 합해 최종 결과를 낸다" 는 개념. W5에서 자세히 다룹니다.

In [ ]:
show_three_axis(bb, title_prefix='BB:')
plt.show()
show_three_axis(cg, title_prefix='CastleGate:')
plt.show()

> **[해석 질문 1]** 세 방향이 "통계적으로는 비슷한 패턴" 이지만 "세부 모양은 완전히 다릅니다".
> 만약 z축 한 방향에서만 보간한다면, 그 방향의 "보간 인공물(어색한 무늬)" 이 그대로 남을 가능성이 큽니다.
> 세 방향에서 보간해서 평균을 낸다면, 이 인공물이 어떻게 처리될까요? (힌트: 평균은 "노이즈" 를 줄인다)

## 4. 공극률 (Porosity, φ) 계산

$$\phi = \frac{\text{공극(pore) voxel 수}}{\text{전체 voxel 수}}$$

본 데이터처럼 값이 0/1뿐이라면, "1의 비율" = "전체 평균" 입니다. 그래서 한 줄로 계산:
```python
phi = volume.mean()
```

**왜 중요한가?** 공극률은 암석의 "빈 공간 비율" → 액체/기체 저장 능력 결정.
본 연구에서는 "보간이 잘 됐는지" 평가할 때 출력의 공극률이 원본과 같아야 합니다.

In [ ]:
phi_bb = porosity(bb)
phi_cg = porosity(cg)
print(f'BB 사암:        φ = {phi_bb:.4f}  ({phi_bb*100:.2f} %)')
print(f'CastleGate 사암: φ = {phi_cg:.4f}  ({phi_cg*100:.2f} %)')

# 막대 그래프로 시각화
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['BB', 'CastleGate'], [phi_bb*100, phi_cg*100], color=[ORANGE, NAVY])
ax.set_ylabel('공극률 φ (%)')
ax.set_title('두 사암의 공극률 비교')
for i, v in enumerate([phi_bb*100, phi_cg*100]):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 슬랩(slab)별 공극률 — 부피가 균일한가?

전체 공극률 한 숫자만 봐서는 "부피의 위쪽과 아래쪽이 다른지" 알 수 없습니다.
부피를 "슬랩(slab)" 이라는 두꺼운 덩어리로 나누어 각 슬랩의 공극률을 보면 됩니다.

> **[보조 설명: slab]** 부피 256×256×256 을 z방향으로 8 슬랩으로 나누면, 한 슬랩은 z 두께 32 voxel × 256 × 256 입니다.

In [ ]:
n_slabs = 8
prof_bb = porosity_profile(bb, axis=0, n_slabs=n_slabs)
prof_cg = porosity_profile(cg, axis=0, n_slabs=n_slabs)

fig, ax = plt.subplots(figsize=(8, 4))
xa = np.arange(n_slabs)
ax.plot(xa, prof_bb*100, marker='o', label='BB', color=ORANGE, lw=2)
ax.plot(xa, prof_cg*100, marker='s', label='CastleGate', color=NAVY, lw=2)
ax.set_xlabel(f'z방향 슬랩 인덱스 (총 {n_slabs}개)')
ax.set_ylabel('공극률 (%)')
ax.set_title('z방향 슬랩별 공극률 — 부피가 균일한지 확인')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **[Try-it! ②]** 위 셀에서 `n_slabs=4, 16, 32` 로 바꿔보세요.
> 슬랩 수가 적으면 곡선이 부드럽고, 많으면 들쭉날쭉합니다. 왜 그럴까요?
>
> 또 `axis=0` 을 `axis=1, axis=2` 로 바꿔서 다른 축으로도 슬랩을 나눠보세요.
> 세 축 결과가 비슷하다면 "부피가 등방성(isotropic)" 입니다.

In [ ]:
# [Try-it! ②] 직접 변형해보세요
for n in [4, 8, 16, 32]:
    p = porosity_profile(bb, axis=0, n_slabs=n)
    print(f'n_slabs={n:2d} → std = {p.std():.5f}')

## 6. Sparse Imaging — 우리 연구의 문제

지금까지는 "완전한 부피" 를 다뤘습니다.

**실제 상황**: micro-CT 스캔은 시간과 비용이 비싸므로, 모든 슬라이스를 다 측정할 수 없을 때가 많습니다.

**해결책 — Sparse imaging**: z축으로 `k` 슬라이스마다 1개만 측정 → 나머지는 컴퓨터로 복원.
- k=1: 전부 측정 (시간 절감 0%)
- k=3: 3장 중 1장만 (시간 67% 절감)
- k=5: 5장 중 1장만 (시간 80% 절감)

**핵심 질문**: "누락된 슬라이스를 측정된 슬라이스로부터 얼마나 정확히 복원할 수 있는가?"

In [ ]:
k = 3
known_idx, missing_idx = make_sparse(bb, k=k, axis=0)

print(f'k={k}: 측정 {len(known_idx)}장 / 누락 {len(missing_idx)}장 / 시간 절감 {time_saving_ratio(k):.1f}%')

# 연속 6 슬라이스에서 어느 것이 측정되고 누락되는지
fig, axes = plt.subplots(1, 6, figsize=(13, 2.8))
for i, z in enumerate(range(60, 66)):
    axes[i].imshow(bb[z])
    is_known = z in known_idx
    status = '측정' if is_known else '누락'
    col = GREEN if is_known else RED
    axes[i].set_title(f'z={z}\n{status}', color=col, fontsize=11)
    axes[i].axis('off')
plt.suptitle(f'Sparse k={k}: 연속 6장 중 어느 슬라이스가 측정되는가', y=1.05)
plt.tight_layout()
plt.show()

## 7. k Sweep — 시간 절감 vs 측정 슬라이스 수

k를 바꿔가며 "시간 절감률" 과 "측정 슬라이스 수" 가 어떻게 변하는지 정리합니다.

> **[Try-it! ③]** 아래 셀의 `k_values` 에 다른 값들을 추가하거나 제거해보세요.
> 어느 k 부터 "측정 슬라이스가 너무 적어" 보간이 어려워 보일까요?

In [ ]:
k_values = [1, 2, 3, 5, 7, 10]

print(f"{'k':>3}  {'측정':>5}  {'누락':>5}  {'시간절감(%)':>10}")
print('-' * 35)
for k in k_values:
    known, missing = make_sparse(bb, k=k, axis=0)
    print(f'{k:>3}  {len(known):>5}  {len(missing):>5}  {time_saving_ratio(k):>10.1f}')

# 시각화
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(k) for k in k_values],
       [time_saving_ratio(k) for k in k_values],
       color=ORANGE)
ax.set_xlabel('k (sparse 간격)')
ax.set_ylabel('시간 절감률 (%)')
ax.set_title('Sparse k에 따른 측정 시간 절감')
for i, k in enumerate(k_values):
    ax.text(i, time_saving_ratio(k)+1, f'{time_saving_ratio(k):.0f}%',
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. 단순 보간 맛보기 — 두 슬라이스 "사이" 만들기

가장 단순한 복원 방법은 **두 측정 슬라이스의 평균** 입니다. 가운데 슬라이스를 "앞 + 뒤의 적당한 비율" 로 추정.

$$\text{예측} = (1-\alpha) \times \text{앞} + \alpha \times \text{뒤}$$

이것이 본 연구의 가장 단순한 baseline ("linear interpolation") 의 핵심 아이디어입니다.
W2에서 정식 구현을, W3 이후 deep learning 버전을 다룹니다.

In [ ]:
# z=60 과 z=66 사이 (간격 6) 의 가운데 (z=63) 슬라이스를 α 값에 따라 만들어보기
z_before, z_after = 60, 66

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, a in zip(axes, alphas):
    pred = linear_interpolate_slice(bb[z_before], bb[z_after], a)
    ax.imshow(pred, vmin=0, vmax=1)
    ax.set_title(f'α={a:.2f}'); ax.axis('off')
plt.suptitle(f'두 슬라이스 (z={z_before} ↔ z={z_after}) 사이의 단순 평균 보간', y=1.05)
plt.tight_layout()
plt.show()

> **[Try-it! ④]** 위 셀의 `z_before, z_after` 의 간격을 1 (인접) → 5 → 15 → 40 으로 바꿔보세요.
> 간격이 클수록 α=0.5 결과(=가운데)는 어떻게 변하나요? (힌트: 흐릿함)
>
> **[해석 질문 2]** 보간 결과는 "0~1 사이 실수" 입니다. 원본은 0/1 binary 였는데요. 왜 그럴까요? 실제 "이진 부피로 복원" 하려면 어떤 후처리가 필요할까요?

## 9. 자기 점검 & 다음 주 예고

### 자기 점검 (자신의 말로 답해보세요)
1. Voxel과 pixel의 차이는?
2. 본 데이터의 "1" 은 무엇을 의미하나요?
3. 공극률은 어떻게 계산하나요?
4. Sparse imaging의 k=3은 시간을 얼마나 절약하나요?
5. "단순 평균 보간" 의 단점은? 왜 deep learning이 필요할까요?

### 다음 주 (W2) 예고
- 본격 선형 / Cubic 보간을 scipy로 구현
- 보간 후 공극률 오차 |Δφ| 직접 측정
- 본 연구의 B1, B2, B3 baseline 만들기

---

## 🎯 W1 탐구 과제 (2조)

**과제 1 (필수)**: 본 노트북의 [Try-it! ①~④] 박스 4개를 모두 직접 수행하고, 각각에 대해 **관찰한 차이를 한 문장씩** 노트에 적어보세요.

**과제 2 (필수)**: BB와 CastleGate 두 도메인에 대해 `porosity_profile(vol, axis=0, n_slabs=16)` 를 각각 실행하고, 두 곡선을 한 plot에 겹쳐 그려보세요. 어느 사암이 "부피가 더 균일한가" 를 판단하고 그 이유를 적어보세요.

**과제 3 (선택)**: `linear_interpolate_slice` 의 α 를 0, 0.1, 0.2, ..., 1.0 (총 11개) 로 sweep 하면서, 각 α에서 "보간 결과의 공극률" 을 계산해보세요. α와 공극률의 관계는 어떤 형태인가요? (예: 직선? 곡선?)

**과제 4 (선택 — 도전)**: `make_sparse` 의 `axis` 인자를 0(z), 1(y), 2(x) 로 바꿔보세요. 같은 k=3이라도 어느 축으로 자르는지에 따라 "측정 슬라이스 분포" 가 달라집니다. 만약 부피가 완벽히 등방성이라면 세 축의 결과는 "통계적으로 같아야 합니다". 직접 확인해보세요.